#  Solucionador Ecuación de Schrödinger — Doble Pozo

Nombre: Juan Sebastián Sanchez Álvarez

Útilizando el mismo potencial de la anterior tarea , se discretiza la E.S en dos métodos principales a comparar: 

 Métodos:
 - Diferencias Finitas 
  $V(x) = 0.1*(x^2 - 10)^2$   
  
- Numerov hbar = m = 1

In [1]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time, warnings
warnings.filterwarnings("ignore")

try:
    import ipywidgets as widgets
    from IPython.display import display
    JUPYTER = True
except ImportError:
    JUPYTER = False

# ── Parámetros globales ──────────────────────────────────────
X_MIN    = -10.0
X_MAX    =  10.0
N_STATES =  6
COLORES  = ["#E63946","#2196F3","#4CAF50","#FF9800","#9C27B0","#00BCD4"]

def V(x):
    return 0.1 * (x**2 - 10.0)**2

# ── Energías analíticas (aprox. armónica local) ──────────────
def energias_analiticas(n_states=6):
    """
    Aproximación: oscilador armónico alrededor de cada mínimo x0=±√10.
    V''(x0) = 8  →  omega = sqrt(8) = 2√2
    E_n^(arm) = omega*(k + 0.5)
    Cada nivel k se desdobla en par simétrico/antisimétrico por efecto túnel.
    El desdoblamiento ΔE ∝ exp(-S_WKB) ≈ 0 para estados bajos.
    """
    omega = 2.0 * np.sqrt(2.0)
    energias = []
    k = 0
    while len(energias) < n_states:
        E = omega * (k + 0.5)
        energias.extend([E, E])   # par cuasi-degenerado
        k += 1
    return np.array(sorted(energias))[:n_states]

# ═══════════════════════════════════════════════════════════════
#  MÉTODO 1: DIFERENCIAS FINITAS  (matriz tridiagonal)
# ═══════════════════════════════════════════════════════════════
def diferencias_finitas(N=600):
    """
    H = -1/2 d²/dx² + V(x) discretizado como tridiagonal:
      diag    =  1/h² + V(xi)
      off_diag= -1/(2h²)
    Autovalores con scipy.linalg.eigh_tridiagonal (O(N) memoria, O(N²) tiempo)
    """
    from scipy.linalg import eigh_tridiagonal

    t0 = time.perf_counter()
    x  = np.linspace(X_MIN, X_MAX, N + 2)[1:-1]
    h  = x[1] - x[0]
    Vi = V(x)

    diag     = 1.0 / h**2 + Vi
    off_diag = np.full(N - 1, -0.5 / h**2)

    vals, vecs = eigh_tridiagonal(diag, off_diag,
                                   select='i', select_range=(0, N_STATES-1))
    for k in range(N_STATES):
        norm = np.trapezoid(vecs[:, k]**2, x)
        vecs[:, k] /= np.sqrt(norm)

    t1 = time.perf_counter()
    return x, vals, vecs, t1 - t0

# ═══════════════════════════════════════════════════════════════
#  MÉTODO 2: NUMEROV con matching izquierda–derecha
# ═══════════════════════════════════════════════════════════════
def _numer_L(E, x, h):
    """Numerov desde frontera izquierda."""
    f = 2.0*(E - V(x)); N = len(x); psi = np.zeros(N)
    psi[0] = 0.0; psi[1] = h
    for i in range(1, N-1):
        d = 1 + h*h/12*f[i+1]
        if abs(d) < 1e-30: break
        psi[i+1] = (2*psi[i]*(1-h*h/12*f[i]) - psi[i-1]*(1+h*h/12*f[i-1])) / d
    return psi

def _numer_R(E, x, h):
    """Numerov desde frontera derecha."""
    f = 2.0*(E - V(x)); N = len(x); psi = np.zeros(N)
    psi[-1] = 0.0; psi[-2] = h
    for i in range(N-2, 0, -1):
        d = 1 + h*h/12*f[i-1]
        if abs(d) < 1e-30: break
        psi[i-1] = (2*psi[i]*(1-h*h/12*f[i]) - psi[i+1]*(1+h*h/12*f[i+1])) / d
    return psi

def _match_fn(E, x, h, mid):
    """Discontinuidad de la derivada logarítmica en x[mid]."""
    pL = _numer_L(E, x, h)
    pR = _numer_R(E, x, h)
    if abs(pL[mid]) < 1e-20 or abs(pR[mid]) < 1e-20:
        return np.nan
    dL = (pL[mid+1]-pL[mid-1])/(2*h*pL[mid])
    dR = (pR[mid+1]-pR[mid-1])/(2*h*pR[mid])
    return dL - dR

def _refine_near(E_seed, x, h, mid, delta=0.05, n_scan=500):
    """
    Barrido fino alrededor de E_seed para encontrar el cero de _match_fn.
    """
    E_lo = max(0.1, E_seed - delta)
    E_hi = E_seed + delta
    Es = np.linspace(E_lo, E_hi, n_scan)
    ms = np.array([_match_fn(E, x, h, mid) for E in Es])

    best = None
    for i in range(len(ms)-1):
        vi, vj = ms[i], ms[i+1]
        if not (np.isfinite(vi) and np.isfinite(vj)): continue
        if vi*vj < 0 and abs(vi) < 300 and abs(vj) < 300:
            Ea, Eb, fa = Es[i], Es[i+1], vi
            for _ in range(100):
                Em = 0.5*(Ea+Eb)
                fm = _match_fn(Em, x, h, mid)
                if not np.isfinite(fm): break
                if fa*fm < 0: Eb = Em
                else: Ea, fa = Em, fm
            best = 0.5*(Ea+Eb)
            break
    return best

def _build_psi(E, x, h, mid):
    """Construye y normaliza la función de onda ensamblando L y R."""
    pL = _numer_L(E, x, h)
    pR = _numer_R(E, x, h)
    if abs(pL[mid]) < 1e-20 or abs(pR[mid]) < 1e-20:
        return None
    pL /= pL[mid]; pR /= pR[mid]
    psi = np.empty_like(pL)
    psi[:mid+1] = pL[:mid+1]
    psi[mid:]   = pR[mid:]
    norm = np.trapezoid(psi**2, x)
    return psi / np.sqrt(norm) if norm > 1e-30 else None

def numerov(N=600, E_seeds=None):
    """
    Método de Numerov con matching izquierda–derecha.
    Usa E_seeds (generalmente los autovalores FD) como punto de partida,
    luego refina con barrido local fino + bisección.
    """
    t0 = time.perf_counter()
    x   = np.linspace(X_MIN, X_MAX, N + 2)[1:-1]
    h   = x[1] - x[0]
    mid = N // 2

    autovalores  = []
    autovectores = []

    if E_seeds is None:
        # Barrido global si no hay semillas
        E_scan = np.linspace(0.3, 12.0, 10000)
        ms = np.array([_match_fn(E, x, h, mid) for E in E_scan])
        E_seeds_list = []
        for i in range(len(ms)-1):
            vi, vj = ms[i], ms[i+1]
            if not (np.isfinite(vi) and np.isfinite(vj)): continue
            if vi*vj < 0 and abs(vi) < 300 and abs(vj) < 300:
                E_seeds_list.append(0.5*(E_scan[i]+E_scan[i+1]))
        E_seeds = E_seeds_list[:N_STATES]
    
    seen = []
    for E_seed in E_seeds:
        # Barrido local muy fino alrededor de la semilla
        E_found = _refine_near(E_seed, x, h, mid, delta=0.08, n_scan=1000)
        if E_found is None:
            # Fallback: usar la semilla directamente si está muy degenerada
            E_found = E_seed
        # Evitar duplicados
        if any(abs(E_found - s) < 0.01 for s in seen):
            # Buscar en lado opuesto del par degenerado
            E_found2 = _refine_near(E_seed + 0.001, x, h, mid, delta=0.08, n_scan=1000)
            if E_found2 and not any(abs(E_found2 - s) < 0.01 for s in seen):
                E_found = E_found2
            else:
                E_found = E_seed  # usar semilla directamente

        psi = _build_psi(E_found, x, h, mid)
        if psi is None:
            # Fallback: Numerov simple desde izquierda
            psi = _numer_L(E_found, x, h)
            norm = np.trapezoid(psi**2, x)
            if norm > 1e-30: psi /= np.sqrt(norm)

        seen.append(E_found)
        autovalores.append(E_found)
        autovectores.append(psi)

    autovalores  = np.array(autovalores)
    autovectores = np.column_stack(autovectores)

    t1 = time.perf_counter()
    return x, autovalores, autovectores, t1 - t0








# ═══════════════════════════════════════════════════════════════
#  TABLA DE RESULTADOS
# ═══════════════════════════════════════════════════════════════

In [3]:

def imprimir_tabla(E_fd, E_num, E_ana, t_fd, t_num):
    print("\n" + "═"*76)
    print("  COMPARACIÓN DE MÉTODOS — Oscilador de Doble Pozo")
    print("═"*76)
    print(f"  {'Estado':>6} │ {'FD (Eₙ)':>10} │ {'Numerov (Eₙ)':>12} │ "
          f"{'Analítica':>10} │ {'Err FD %':>8} │ {'Err Num %':>9}")
    print("─"*76)
    for i in range(N_STATES):
        Ea = E_ana[i] if i < len(E_ana) else np.nan
        Ef = E_fd[i]  if i < len(E_fd)  else np.nan
        En = E_num[i] if i < len(E_num) else np.nan
        ef = abs(Ef-Ea)/abs(Ea)*100 if np.isfinite(Ea) and Ea!=0 else np.nan
        en = abs(En-Ea)/abs(Ea)*100 if np.isfinite(Ea) and Ea!=0 and np.isfinite(En) else np.nan
        print(f"  {'n='+str(i):>6} │ {Ef:>10.6f} │ {En:>12.6f} │ "
              f"{Ea:>10.6f} │ {ef:>7.3f}% │ {en:>8.3f}%")
    print("═"*76)
    print(f"  ⏱  Diferencias Finitas : {t_fd*1000:7.2f} ms")
    print(f"  ⏱  Numerov (matching)  : {t_num*1000:7.2f} ms")
    print(f"  ⏱  Factor de velocidad : ×{t_num/t_fd:.1f} más lento Numerov")
    print("═"*76 + "\n")

# ═══════════════════════════════════════════════════════════════
#  GRÁFICO ESTÁTICO  (modo terminal)
# ═══════════════════════════════════════════════════════════════

In [4]:

def grafico_estatico(x_fd, E_fd, vf, x_num, E_num, vn, E_ana, t_fd, t_num):
    fig = plt.figure(figsize=(18,14), facecolor="#0d1117")
    fig.suptitle("Ecuación de Schrödinger — Oscilador de Doble Pozo\n"
                 r"$V(x)=0.1\,(x^2-10)^2$, unidades naturales $\hbar=m=1$",
                 color="white", fontsize=15, fontweight="bold", y=0.98)

    gs = gridspec.GridSpec(2,3, figure=fig,
                           hspace=0.42, wspace=0.35,
                           left=0.07, right=0.97, top=0.91, bottom=0.07)
    ax1 = fig.add_subplot(gs[0,:2])
    ax2 = fig.add_subplot(gs[1,:2])
    ax3 = fig.add_subplot(gs[:,2])

    xp  = np.linspace(X_MIN, X_MAX, 1000)
    Vp  = V(xp)
    esc = 0.4

    for ax, x_, E_, vecs_, titulo in [
        (ax1, x_fd, E_fd, vf, "Diferencias Finitas"),
        (ax2, x_num, E_num, vn, "Numerov (matching L↔R)")
    ]:
        ax.set_facecolor("#0d1117")
        ax.plot(xp, Vp, color="white", lw=1.5, alpha=0.4, label="V(x)")
        for i in range(min(N_STATES, vecs_.shape[1])):
            E  = E_[i]
            p2 = vecs_[:,i]**2
            ax.hlines(E, X_MIN, X_MAX, colors=COLORES[i],
                      linestyles="--", lw=0.8, alpha=0.6)
            ax.fill_between(x_, E, p2*esc+E, alpha=0.30, color=COLORES[i])
            ax.plot(x_, p2*esc+E, color=COLORES[i], lw=1.4,
                    label=f"n={i}  E={E:.4f}")
        ax.set_xlim(X_MIN, X_MAX); ax.set_ylim(-0.5, 8.0)
        ax.set_title(titulo, color="white", fontsize=12, pad=6)
        ax.set_xlabel("x", color="white"); ax.set_ylabel("Energía", color="white")
        ax.tick_params(colors="white"); ax.spines[:].set_color("#444")
        ax.legend(fontsize=8, loc="upper right",
                  facecolor="#1a1a2e", labelcolor="white", framealpha=0.85)

    # Panel de error
    ax3.set_facecolor("#0d1117")
    estados = np.arange(N_STATES)
    n_num   = len(E_num)
    err_fd  = [abs(E_fd[i]-E_ana[i])/abs(E_ana[i])*100 for i in range(N_STATES)]
    err_num = [abs(E_num[i]-E_ana[i])/abs(E_ana[i])*100
               for i in range(min(N_STATES,n_num)) if np.isfinite(E_num[i])]

    bw = 0.35
    ax3.barh(estados - bw/2, err_fd, height=bw, color="#E63946",
             alpha=0.85, label="Dif. Finitas")
    if err_num:
        ax3.barh(estados[:len(err_num)]+bw/2, err_num, height=bw,
                 color="#2196F3", alpha=0.85, label="Numerov")

    # Tiempos como texto
    ax3.text(0.98, 0.02,
             f"FD:  {t_fd*1000:.1f} ms\nNumerov: {t_num*1000:.1f} ms\n"
             f"(×{t_num/t_fd:.0f} más lento)",
             transform=ax3.transAxes, ha='right', va='bottom',
             color="#aaa", fontsize=9,
             bbox=dict(facecolor="#1a1a2e", edgecolor="#444", boxstyle="round,pad=0.4"))

    ax3.set_yticks(estados)
    ax3.set_yticklabels([f"n={i}" for i in estados], color="white")
    ax3.set_xlabel("Error relativo (%)", color="white")
    ax3.set_title("Error vs. aprox. analítica\n(oscilador armónico local)",
                  color="white", fontsize=11)
    ax3.tick_params(colors="white"); ax3.spines[:].set_color("#444")
    ax3.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=9)

    plt.savefig("/mnt/user-data/outputs/schrodinger_plot.png",
                dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()

# ═══════════════════════════════════════════════════════════════
#  GRÁFICO INTERACTIVO  (modo Jupyter + ipywidgets)
# ═══════════════════════════════════════════════════════════════

In [6]:


def grafico_interactivo(x_fd, E_fd, vf, x_num, E_num, vn, E_ana, t_fd, t_num):
    xp = np.linspace(X_MIN, X_MAX, 1000)
    Vp = V(xp)

    def actualizar(estado, metodo, x_min, x_max, escala):
        fig, axes = plt.subplots(1,2, figsize=(16,6), facecolor="#0d1117")
        fig.suptitle(
            f"Estado n={estado}  —  {metodo}\n"
            rf"$V(x)=0.1\,(x^2-10)^2$",
            color="white", fontsize=13, fontweight="bold")

        # ── Panel izquierdo: todos los estados ──
        ax = axes[0]; ax.set_facecolor("#0d1117")
        ax.plot(xp, Vp, "w-", lw=1.2, alpha=0.3, label="V(x)")
        xs, Es, Vs = (x_fd, E_fd, vf) if metodo=="Diferencias Finitas" \
                     else (x_num, E_num, vn)
        for i in range(min(N_STATES, Vs.shape[1])):
            E = Es[i]; p2 = Vs[:,i]**2
            a  = 1.0 if i==estado else 0.2
            lw = 2.0 if i==estado else 0.7
            ax.hlines(E, x_min, x_max, colors=COLORES[i],
                      linestyles="--", lw=0.7, alpha=a*0.8)
            ax.fill_between(xs, E, p2*escala+E, alpha=a*0.3, color=COLORES[i])
            ax.plot(xs, p2*escala+E, color=COLORES[i], lw=lw,
                    label=f"n={i}  E={E:.4f}")
        ax.set_xlim(x_min, x_max); ax.set_ylim(-0.5, 8.5)
        ax.set_xlabel("x", color="w"); ax.set_ylabel("Energía", color="w")
        ax.set_title("Todos los estados", color="w")
        ax.tick_params(colors="w"); ax.spines[:].set_color("#444")
        ax.legend(fontsize=8, facecolor="#1a1a2e", labelcolor="w", framealpha=0.8)

        # ── Panel derecho: estado seleccionado, comparación FD vs Numerov ──
        ax2 = axes[1]; ax2.set_facecolor("#0d1117")
        ax2.plot(xp, Vp, "w-", lw=1.2, alpha=0.3, label="V(x)")

        E_f = E_fd[estado];  p2_f = vf[:,estado]**2
        ax2.plot(x_fd, p2_f*escala+E_f, color="#E63946", lw=2,
                 label=f"FD  E={E_f:.5f}")
        ax2.fill_between(x_fd, E_f, p2_f*escala+E_f, alpha=0.2, color="#E63946")
        ax2.hlines(E_f, x_min, x_max, colors="#E63946", linestyles="--", lw=1)

        if estado < vn.shape[1]:
            E_n = E_num[estado]; p2_n = vn[:,estado]**2
            ax2.plot(x_num, p2_n*escala+E_n, color="#2196F3", lw=2,
                     label=f"Numerov  E={E_n:.5f}")
            ax2.fill_between(x_num, E_n, p2_n*escala+E_n, alpha=0.2, color="#2196F3")
            ax2.hlines(E_n, x_min, x_max, colors="#2196F3", linestyles="--", lw=1)

        E_a = E_ana[estado]
        ax2.hlines(E_a, x_min, x_max, colors="#4CAF50", linestyles=":", lw=2,
                   label=f"Analítica  E={E_a:.5f}")

        E_c = E_fd[estado]
        ax2.set_xlim(x_min, x_max); ax2.set_ylim(E_c-0.5, E_c+1.4)
        ax2.set_xlabel("x", color="w")
        ax2.set_ylabel(r"$E_n + |\psi_n|^2\cdot$escala", color="w")
        ax2.set_title(f"Detalle n={estado}  (FD  vs  Numerov  vs  Analítica)",
                      color="w")
        ax2.tick_params(colors="w"); ax2.spines[:].set_color("#444")
        ax2.legend(fontsize=9, facecolor="#1a1a2e", labelcolor="w", framealpha=0.9)

        fig.text(0.01, 0.01,
                 f"⏱ FD: {t_fd*1000:.1f} ms   |   Numerov: {t_num*1000:.1f} ms   "
                 f"|   Numerov es ×{t_num/t_fd:.0f} más lento",
                 color="#999", fontsize=9)
        plt.tight_layout(); plt.show()

    sl = dict(continuous_update=False, layout=widgets.Layout(width="420px"))
    w_estado = widgets.IntSlider(value=0, min=0, max=N_STATES-1, step=1,
                                  description="Estado:", **sl)
    w_metodo = widgets.ToggleButtons(
        options=["Diferencias Finitas","Numerov"],
        description="Método:",
        style={"button_width":"160px"})
    w_xmin = widgets.FloatSlider(value=X_MIN, min=X_MIN, max=0,  step=0.2,
                                  description="x min:", **sl)
    w_xmax = widgets.FloatSlider(value=X_MAX, min=0,    max=X_MAX,step=0.2,
                                  description="x max:", **sl)
    w_esc  = widgets.FloatSlider(value=0.4,  min=0.05, max=2.0, step=0.05,
                                  description="Escala ψ²:", **sl)

    ui = widgets.VBox([
        widgets.HTML("<h3 style='color:#aaa;font-family:monospace'>"
                     "🔬 Schrödinger — Doble Pozo Cuántico</h3>"),
        w_metodo,
        widgets.HBox([w_estado, w_esc]),
        widgets.HBox([w_xmin,   w_xmax]),
    ])
    out = widgets.interactive_output(
        actualizar,
        {"estado":w_estado,"metodo":w_metodo,
         "x_min":w_xmin,"x_max":w_xmax,"escala":w_esc})
    display(ui, out)

# ═══════════════════════════════════════════════════════════════
#  PROGRAMA PRINCIPAL
# ═══════════════════════════════════════════════════════════════

In [7]:

def main():
    print("╔═══════════════════════════════════════════════════╗")
    print("║  Schrödinger — Oscilador de Doble Pozo           ║")
    print("║  V(x) = 0.1·(x²-10)²   |   ħ = m = 1           ║")
    print("╚═══════════════════════════════════════════════════╝\n")

    N = 600   # aumentar para mayor precisión (ej. 1000, 2000)

    print(f"Resolviendo con N={N} puntos de malla...\n")

    print("  [1/3] Diferencias Finitas (FD)...", end=" ", flush=True)
    x_fd, E_fd, vf, t_fd = diferencias_finitas(N)
    print(f"✓  ({t_fd*1000:.1f} ms)")

    print("  [2/3] Numerov (matching bidireccional)...", end=" ", flush=True)
    x_num, E_num, vn, t_num = numerov(N, E_seeds=E_fd.tolist())
    print(f"✓  ({t_num*1000:.1f} ms)")

    print("  [3/3] Energías analíticas (aprox. OA local)...", end=" ", flush=True)
    E_ana = energias_analiticas(N_STATES)
    print("✓\n")

    imprimir_tabla(E_fd, E_num, E_ana, t_fd, t_num)

    if JUPYTER:
        print("Modo Jupyter → widgets interactivos\n")
        grafico_interactivo(x_fd, E_fd, vf, x_num, E_num, vn, E_ana, t_fd, t_num)
    else:
        print("Modo terminal → gráfico estático\n")
        grafico_estatico(x_fd, E_fd, vf, x_num, E_num, vn, E_ana, t_fd, t_num)

if __name__ == "__main__":
    main()

╔═══════════════════════════════════════════════════╗
║  Schrödinger — Oscilador de Doble Pozo           ║
║  V(x) = 0.1·(x²-10)²   |   ħ = m = 1           ║
╚═══════════════════════════════════════════════════╝

Resolviendo con N=600 puntos de malla...

  [1/3] Diferencias Finitas (FD)... ✓  (2.5 ms)
  [2/3] Numerov (matching bidireccional)... ✓  (16316.1 ms)
  [3/3] Energías analíticas (aprox. OA local)... ✓


════════════════════════════════════════════════════════════════════════════
  COMPARACIÓN DE MÉTODOS — Oscilador de Doble Pozo
════════════════════════════════════════════════════════════════════════════
  Estado │    FD (Eₙ) │ Numerov (Eₙ) │  Analítica │ Err FD % │ Err Num %
────────────────────────────────────────────────────────────────────────────
     n=0 │   1.387858 │     1.387858 │   1.414214 │   1.864% │    1.864%
     n=1 │   1.387858 │     1.387858 │   1.414214 │   1.864% │    1.864%
     n=2 │   4.049090 │     4.049090 │   4.242641 │   4.562% │    4.562%
     n=3 │

Output()